# Table 4 Ablation - Duplicate Query Context (SUN397, strict pair-concat)

Notebook này tái hiện hàng `Duplicate Query` trong Table 4.

Context image không được retrieve từ candidate pool. Mỗi query được ghép với cùng raw image của chính nó, nhưng query/context được transform như hai input riêng biệt. Điều này tránh shortcut `context = query.clone()` sau augmentation, vì clone tensor làm mô hình nhìn đúng cùng token hai lần và có thể đẩy metric lên bất thường.

Model vẫn dùng strict token-level pair concatenation: query tokens + duplicate-query-context tokens đi qua ViT blocks rồi classifier đọc CLS.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !unzip -q /content/drive/MyDrive/archive\ \(13\).zip -d /content/archive\ \(13\)/

## 1. Cài thư viện

Chạy cell này trên Colab nếu môi trường chưa có đủ thư viện. Nếu chạy local đã cài sẵn thì có thể bỏ qua.


In [ ]:
%pip -q install torch torchvision pandas pillow tqdm matplotlib torchmetrics


In [ ]:
# !wget -c 'https://dinov3.llamameta.net/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiNTFma3Jxa3l3bTl3MGpweHI2NndvNWZsIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3ODkyOTc0MTJ9fX1dfQ__&Signature=sdowk%7EiiF6j8599sB9K5-M3sB%7Elq1be-oKP%7El7VBISEmjHEBPyCWw9Y2JD4oJY9LbXE313Q4ZgjX%7EXO3EiwGQZ3MZhfkZr0OXmADX8L%7E7XX%7E4mkVA1JSOkVzEa3Oj39gY%7E%7ER21%7EmJXEcFOZPMj%7EzGs-uxIh1xeRrfo-ZUqv3Yz7z7vUgn1OoDhu98kPnh8KT4MF1Gd6gcB4LogqWk3ooVScR6z8ECw50eawfY8lyUUzv%7E%7Em1QdstSU1k6HFhKFbuuogf0MyH7rzYScskxtGG95ke47CKn6CWSNbHLKhcLDfNx-XrN0A-xxis10rpJaMh2jBmxwohSr8XQlGVCBPWqQ__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1749739516259745' \
#   -O /content/dinov3_vits16_pretrain_lvd1689m-08c60483.pth

## 2. Import, cấu hình, dataset và candidate pool

Cell này đọc standard SUN397 split nếu có `Training_01.txt` / `Testing_01.txt`; nếu không có thì tự tạo split 50 train / 50 test mỗi class. Candidate pool lấy **20% train split** như paper.

Bản này không dùng random shortlist nữa. Trong train/eval, mỗi query được tính utility score với toàn bộ candidate pool.


In [ ]:
import json
import random
import subprocess
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
# Tận dụng Tensor Cores thế hệ mới trên dòng RTX / GPU Ampere+
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

# --- ĐƯỜNG DẪN OUTPUT (Ghi vào working của Kaggle) ---
PROJECT_DIR = Path('/kaggle/working')
OUT_PUT_DIR = PROJECT_DIR / 'outputs' / 'table4_duplicate_query_sun_independent_aug_pairconcat'
OUTPUT_DIR = OUT_PUT_DIR
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

# --- ĐƯỜNG DẪN DATASET (SUN397) ---
DATASET_ROOT = Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397')
if not DATASET_ROOT.exists():
    fallback_candidates = [
        Path('/kaggle/input/sun397/SUN397/SUN397'),
        Path('/kaggle/input/sun397/SUN397'),
        Path('/kaggle/input/sun397'),
        Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397'),
    ]
    for candidate in fallback_candidates:
        if candidate.exists():
            DATASET_ROOT = candidate
            break

IMAGE_ROOT = DATASET_ROOT

# --- ĐƯỜNG DẪN DINOv3 REPO & WEIGHTS ---
DINOV3_REPO_DIR = '/kaggle/input/datasets/nguyenducxu/dinov3-code/dinov3-main'
if not Path(DINOV3_REPO_DIR).exists():
    for p in [Path('/kaggle/input/dinov3-code/dinov3-main'), Path('/kaggle/input/dinov3-code')]:
        if p.exists():
            DINOV3_REPO_DIR = str(p)
            break

DINOV3_MODEL_NAME = 'dinov3_vits16'
DINOV3_WEIGHTS = '/kaggle/input/datasets/nguyenducxu/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
if not Path(DINOV3_WEIGHTS).exists():
    for p in [
        Path('/kaggle/input/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'),
        Path('/kaggle/input/dinov3/dinov3_vits16.pth'),
    ]:
        if p.exists():
            DINOV3_WEIGHTS = str(p)
            break

BACKBONE_NAME = DINOV3_MODEL_NAME
FEATURE_DIM = 384

IMAGE_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
BACKBONE_LR = 3e-6
HEAD_LR = 1e-4
BACKBONE_WEIGHT_DECAY = 0.05
HEAD_WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
TOKEN_HEAD_DROPOUT = 0.3
FREEZE_BACKBONE_EPOCHS = 0
WARMUP_EPOCHS = 5
PATIENCE = 10
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 8
NUM_CLASSES = 397  # SUN397 có 397 scene categories
TOP_K = 5
CONTEXT_TOP_K = 1
RETRIEVAL_SCORE_WEIGHT = 0.0
SUPCON_WEIGHT = 0.05
SUPCON_TEMPERATURE = 0.10
USE_BBOX_CROP = False  # SUN397 là bài toán scene classification, không dùng crop bbox
CANDIDATE_POOL_FRACTION = 0.20
VAL_FRACTION = 0.10
RETRIEVAL_BATCH_SIZE = 512
RETRIEVAL_CHUNK_SIZE = 4096
QUERY_CONTEXT_GATE_DROPOUT = 0.20
QUERY_CONTEXT_GATE_BIAS = -0.7
EVAL_TTA_HORIZONTAL_FLIP = False
USE_CONTEXT_LABEL_PRIOR = False
CONTEXT_LABEL_PRIOR_WEIGHT = 0.25
CONTEXT_LABEL_PRIOR_TEMPERATURE = 0.07

TRAIN_ON_FULL_TRAIN = False
FULL_TRAIN_MONITOR_FRACTION = 0.05  # chi dung khi TRAIN_ON_FULL_TRAIN=True

# --- Table 4 context ablation hyperparameters ---
CONTEXT_ABLATION_METHOD = 'duplicate_query'

# --- TACS selector hyperparameters (giu lai de so sanh config, khong dung trong Duplicate Query) ---
TACS_CANDIDATES_PER_QUERY = None  # None nghĩa là dùng toàn bộ candidate pool
TACS_PREFILTER_TOP_SIMILAR = 0
TACS_GUMBEL_TAU = 0.10
TACS_POLICY_WEIGHT = 1.0
TACS_ENTROPY_WEIGHT = 0.01
TACS_SELECTOR_LR = 1e-4
TACS_SELECTOR_WEIGHT_DECAY = 1e-4
TACS_SELECTOR_DROPOUT = 0.10
TACS_SELECTOR_TEMPERATURE = 1.0

# --- Full-pool selector memory controls ---
TACS_USE_FULL_CANDIDATE_POOL = True
CANDIDATE_IMAGE_BANK_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CANDIDATE_IMAGE_BANK_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
CANDIDATE_FEATURE_BANK_BATCH_SIZE = 128
FULL_POOL_SCORE_CHUNK_SIZE = 1024

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 48, IMAGE_SIZE + 48)),
    transforms.RandomResizedCrop(
        (IMAGE_SIZE, IMAGE_SIZE),
        scale=(0.60, 1.0),
        ratio=(0.80, 1.25),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.25,
            contrast=0.25,
            saturation=0.20,
            hue=0.04,
        )
    ], p=0.75),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
    ], p=0.15),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.20,
        scale=(0.02, 0.12),
        ratio=(0.3, 3.3),
        value="random",
    ),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

_mean = globals().get("DINO_MEAN", globals().get("IMAGENET_MEAN", (0.485, 0.456, 0.406)))
_std = globals().get("DINO_STD", globals().get("IMAGENET_STD", (0.229, 0.224, 0.225)))

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=_mean, std=_std),
])


def load_class_names(dataset_root):
    classes_path = dataset_root / 'ClassName.txt'
    if not classes_path.is_file():
        fallback_paths = [
            dataset_root.parent / 'ClassName.txt',
            dataset_root / 'classes.txt',
            Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397/ClassName.txt'),
            Path('/kaggle/input/sun397/SUN397/SUN397/ClassName.txt'),
        ]
        for p in fallback_paths:
            if p.is_file():
                classes_path = p
                break

    if not classes_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy ClassName.txt tại: {classes_path}")

    class_names = []
    with open(classes_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                # Dòng có dạng /a/abbey hoặc a/abbey -> chuẩn hóa bỏ dấu / ở đầu
                class_names.append(line.lstrip('/'))

    global NUM_CLASSES
    if len(class_names) != NUM_CLASSES:
        print(f"Cập nhật NUM_CLASSES từ {NUM_CLASSES} thành {len(class_names)}")
        NUM_CLASSES = len(class_names)
    return class_names


def load_sun397_annotations(dataset_root, image_root, class_names):
    class_to_id = {cls.lstrip('/'): idx for idx, cls in enumerate(class_names)}

    # 1. Tìm các file partition chuẩn (Training_01.txt / Testing_01.txt) nếu có sẵn trong dataset
    train_part_file, test_part_file = None, None
    search_dirs = [
        dataset_root,
        dataset_root.parent / 'sun397_other',
        dataset_root.parent / 'sun397_other' / 'Partitions',
        dataset_root.parent.parent / 'sun397_other',
        Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/sun397_other'),
    ]
    for d in search_dirs:
        if d.is_dir():
            t1 = d / 'Training_01.txt'
            te1 = d / 'Testing_01.txt'
            if t1.is_file() and te1.is_file():
                train_part_file, test_part_file = t1, te1
                break
            found_t = list(d.glob('**/Training_01.txt'))
            if found_t:
                train_part_file = found_t[0]
                cand_te = train_part_file.parent / 'Testing_01.txt'
                if cand_te.is_file():
                    test_part_file = cand_te
                    break

    records = []
    if train_part_file and test_part_file:
        print(f"Sử dụng official partition: {train_part_file.name} và {test_part_file.name}")
        for split_label, p_file in [('train', train_part_file), ('test', test_part_file)]:
            with open(p_file, 'r', encoding='utf-8') as f:
                for line in f:
                    rel_p = line.strip().lstrip('/')
                    if not rel_p:
                        continue
                    class_folder = str(Path(rel_p).parent).replace('\\', '/').lstrip('/')
                    if class_folder not in class_to_id:
                        continue
                    class_id = class_to_id[class_folder]
                    full_p = image_root / rel_p
                    if not full_p.is_file():
                        if (dataset_root / rel_p).is_file():
                            full_p = dataset_root / rel_p
                        else:
                            continue
                    records.append({
                        'rel_path': rel_p,
                        'fname': Path(rel_p).name,
                        'image_path': str(full_p.resolve()),
                        'class_id': int(class_id),
                        'is_train': 1 if split_label == 'train' else 0,
                    })
        df = pd.DataFrame(records)
    else:
        print("Không thấy file partition, tự động quét thư mục ảnh theo ClassName.txt và chia split 50/50 chuẩn...")
        all_rows = []
        rng = np.random.default_rng(SEED)
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp'}

        for cls_name in tqdm(class_names, desc="Đang đọc các lớp SUN397"):
            cls_dir = image_root / cls_name
            if not cls_dir.is_dir():
                cls_dir = dataset_root / cls_name
            if not cls_dir.is_dir():
                continue

            cls_id = class_to_id[cls_name]
            img_paths = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_exts]
            img_paths.sort()
            if not img_paths:
                continue

            n_total = len(img_paths)
            indices = np.arange(n_total)
            rng.shuffle(indices)

            # Benchmark chuẩn SUN397: 50 train / 50 test mỗi class nếu >= 100 ảnh, hoặc chia đôi
            if n_total >= 100:
                n_train = 50
                n_test = 50
                train_idx = set(indices[:n_train])
                test_idx = set(indices[n_train:n_train + n_test])
            else:
                n_train = max(1, n_total // 2)
                train_idx = set(indices[:n_train])
                test_idx = set(indices[n_train:])

            for idx, p in enumerate(img_paths):
                if idx in train_idx:
                    is_train = 1
                elif idx in test_idx:
                    is_train = 0
                else:
                    continue

                try:
                    rel_p = str(p.relative_to(image_root)).replace('\\', '/')
                except Exception:
                    rel_p = f"{cls_name}/{p.name}"

                all_rows.append({
                    'rel_path': rel_p,
                    'fname': p.name,
                    'image_path': str(p.resolve()),
                    'class_id': int(cls_id),
                    'is_train': is_train,
                })
        df = pd.DataFrame(all_rows)

    # Thêm các cột dummy BBox để tương thích 100% với Dataset class ở Cell 3 mà không gây lỗi
    df['image_id'] = np.arange(len(df))
    df['bbox_x1'] = 0
    df['bbox_y1'] = 0
    df['bbox_x2'] = 99999
    df['bbox_y2'] = 99999

    train_df = df[df['is_train'] == 1].copy().reset_index(drop=True)
    train_df['split'] = 'train'
    train_df['row_id'] = np.arange(len(train_df))

    test_df = df[df['is_train'] == 0].copy().reset_index(drop=True)
    test_df['split'] = 'test'
    test_df['row_id'] = np.arange(len(test_df))

    return train_df, test_df


def split_train_val_by_class(df, val_fraction=0.10, seed=42):
    if not 0 < val_fraction < 1:
        raise ValueError('val_fraction phải nằm trong khoảng (0, 1)')
    rng = np.random.default_rng(seed)
    val_indices = []
    for _, group in df.groupby('class_id', sort=True):
        group_indices = group.index.to_numpy()
        if len(group_indices) <= 1:
            continue
        n_val = max(1, int(round(len(group_indices) * val_fraction)))
        n_val = min(n_val, len(group_indices) - 1)
        val_indices.extend(rng.choice(group_indices, size=n_val, replace=False).tolist())
    val_indices = np.array(sorted(val_indices), dtype=int)
    train_split = df.drop(index=val_indices).reset_index(drop=True)
    val_split = df.loc[val_indices].reset_index(drop=True)
    return train_split, val_split


# --- Tải dữ liệu SUN397 ---
class_names = load_class_names(DATASET_ROOT)
full_train_df, test_df = load_sun397_annotations(DATASET_ROOT, IMAGE_ROOT, class_names)
train_df, val_df = split_train_val_by_class(full_train_df, VAL_FRACTION, SEED)

if TRAIN_ON_FULL_TRAIN:
    _, monitor_val_df = split_train_val_by_class(
        full_train_df,
        FULL_TRAIN_MONITOR_FRACTION,
        SEED,
    )
    train_df = full_train_df.reset_index(drop=True)
    val_df = monitor_val_df.reset_index(drop=True)

candidate_rng = np.random.default_rng(SEED)
if CANDIDATE_POOL_FRACTION >= 1.0:
    candidate_pool_positions = np.arange(len(train_df), dtype=int)
else:
    candidate_pool_size = max(1, int(round(len(train_df) * CANDIDATE_POOL_FRACTION)))
    candidate_pool_positions = np.sort(
        candidate_rng.choice(len(train_df), size=candidate_pool_size, replace=False)
    )

candidate_df = train_df.iloc[candidate_pool_positions].reset_index(drop=True)
candidate_df.to_csv(OUT_PUT_DIR / 'candidate_pool.csv', index=False)
val_df.to_csv(OUT_PUT_DIR / 'validation_split.csv', index=False)

print('DATASET_ROOT:', DATASET_ROOT.resolve())
print('IMAGE_ROOT:', IMAGE_ROOT.resolve())
print('OUT_PUT_DIR:', OUT_PUT_DIR.resolve())
print(f'Standard train images: {len(full_train_df):,}')
print(f'Actual train images: {len(train_df):,}')
print(f'Validation images: {len(val_df):,}')
print(f'Train on full train: {TRAIN_ON_FULL_TRAIN}')
print(f'Standard test images: {len(test_df):,}')
print(f'Candidate pool: {len(candidate_df):,}/{len(train_df):,} actual-train images ({CANDIDATE_POOL_FRACTION:.0%})')
print(f'Table 4 context method: {CONTEXT_ABLATION_METHOD}')
print('Context construction: same raw image as query, transformed through a separate context pipeline')
print('Candidate pool is saved for table bookkeeping but not used by Duplicate Query baseline.')
print(f'BBox crop enabled: {USE_BBOX_CROP}')
print(f'Train label range: {int(train_df.class_id.min())}..{int(train_df.class_id.max())}')
print(f'Test label range: {int(test_df.class_id.min())}..{int(test_df.class_id.max())}')
display(train_df.head())
display(val_df.head())
display(candidate_df.head())

## 3. Dataset và DataLoader

Với full candidate pool, train dataset chỉ cần trả về query image và label. Candidate images được cache riêng trong candidate image bank; selector sẽ tính score với toàn bộ candidate pool trong train loop.


In [ ]:
class SimpleImageDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = crop_bbox

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        image = self.transform(self._open_image(row))
        return image, int(index), row['fname'], int(row['class_id'])


class StanfordCarsSimilarityContextDataset(Dataset):
    def __init__(self, annotations, candidate_annotations, context_positions, context_scores, query_transform, context_transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.candidate_annotations = candidate_annotations.reset_index(drop=True).copy()
        self.context_positions = np.asarray(context_positions, dtype=np.int64)
        self.context_scores = np.asarray(context_scores, dtype=np.float32)
        if self.context_positions.ndim == 1:
            self.context_positions = self.context_positions[:, None]
        if self.context_scores.ndim == 1:
            self.context_scores = self.context_scores[:, None]
        self.query_transform = query_transform
        self.context_transform = context_transform
        self.crop_bbox = crop_bbox
        if len(self.context_positions) != len(self.annotations):
            raise ValueError('context_positions phai co cung do dai voi annotations')
        if len(self.context_scores) != len(self.annotations):
            raise ValueError('context_scores phai co cung do dai voi annotations')

    def __len__(self):
        return len(self.annotations)

    def set_epoch(self, epoch):
        pass

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        query_row = self.annotations.iloc[index]
        context_positions = self.context_positions[index]
        context_scores = self.context_scores[index]
        context_rows = [self.candidate_annotations.iloc[int(pos)] for pos in context_positions]
        query_image = self.query_transform(self._open_image(query_row))
        context_images = torch.stack([self.context_transform(self._open_image(context_row)) for context_row in context_rows], dim=0)
        context_labels = torch.tensor([int(context_row['class_id']) for context_row in context_rows], dtype=torch.long)
        return {
            'query_image': query_image,
            'context_images': context_images,
            'label': torch.tensor(int(query_row['class_id']), dtype=torch.long),
            'fname': query_row['fname'],
            'context_fnames': '|'.join(str(context_row['fname']) for context_row in context_rows),
            "context_labels": torch.tensor(context_labels, dtype=torch.long),
            'context_scores': torch.as_tensor(context_scores, dtype=torch.float32),
        }




class StanfordCarsTACSQueryDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = crop_bbox

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        return {
            'query_image': self.transform(self._open_image(row)),
            'label': torch.tensor(int(row['class_id']), dtype=torch.long),
            'fname': row['fname'],
            'row_id': torch.tensor(int(row['row_id']), dtype=torch.long),
        }


## 4. Duplicate Query Pair-Concatenation Model

Đây là ablation sạch cho hàng `Duplicate Query`: không selector, không candidate pool trong forward. Context là cùng raw image với query, nhưng được transform riêng để tránh shortcut nhân đôi tensor y hệt.


In [ ]:
from pathlib import Path
import gc
import torch


def resolve_dinov3_weights():
    path = Path(DINOV3_WEIGHTS)
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy file weight: {path}')
    return str(path)


def load_dinov3_model(repo_dir, model_name, weights_path):
    repo_dir = Path(repo_dir)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy thư mục mã nguồn: {repo_dir}')
    weights_path = Path(weights_path)
    if not weights_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy file weight: {weights_path}')
    return torch.hub.load(str(repo_dir), model_name, source='local', weights=str(weights_path))


def load_dinov3_backbone():
    return load_dinov3_model(DINOV3_REPO_DIR, DINOV3_MODEL_NAME, resolve_dinov3_weights())


def ensure_dinov3_assets():
    resolve_dinov3_weights()
    repo_dir = Path(DINOV3_REPO_DIR)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy thư mục mã nguồn: {repo_dir}')
    print('DINOv3 assets đã sẵn sàng từ Kaggle Input.')


def extract_cls_features(backbone, images):
    features = backbone(images)
    if isinstance(features, dict):
        if 'x_norm_clstoken' in features:
            return features['x_norm_clstoken']
        if 'x_norm_cls_token' in features:
            return features['x_norm_cls_token']
        if 'x_norm_patchtokens' in features:
            return features['x_norm_patchtokens'].mean(dim=1)
        raise KeyError(f'Khong tim thay CLS token trong output keys: {list(features)}')
    return features[:, 0] if torch.is_tensor(features) and features.ndim == 3 else features


def extract_frozen_dino_embeddings(annotations, backbone, desc):
    dataset = SimpleImageDataset(annotations, eval_transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=RETRIEVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,persistent_workers=True if NUM_WORKERS > 0 else False)
    embeddings = []
    backbone.eval()
    for parameter in backbone.parameters():
        parameter.requires_grad_(False)
    with torch.inference_mode():
        for images, *_ in tqdm(loader, desc=desc, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                features = extract_cls_features(backbone, images)
            features = F.normalize(features.float(), dim=1)
            embeddings.append(features.cpu())
    return torch.cat(embeddings, dim=0)




class DinoV3FrozenSimilarityContextClassifier(nn.Module):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=384, allow_feature_fallback=False):
        super().__init__()
        self.backbone = load_dinov3_model(repo_dir, model_name, weights_path)
        self.feature_dim = feature_dim
        self.allow_feature_fallback = allow_feature_fallback
        self.active_fusion_mode = 'topk_token_attention'
        self.context_proj = nn.Linear(feature_dim, feature_dim)
        self.query_proj = nn.Linear(feature_dim, feature_dim)
        self.context_attention = nn.Linear(feature_dim, 1)
        self.fusion_norm = nn.LayerNorm(feature_dim)
        self.token_head = nn.Sequential(nn.Dropout(p=TOKEN_HEAD_DROPOUT), nn.Linear(feature_dim, num_classes))
        self.feature_fusion = nn.Sequential(nn.Linear(feature_dim * 2, feature_dim), nn.GELU(), nn.Dropout(p=TOKEN_HEAD_DROPOUT), nn.Linear(feature_dim, num_classes))
        for param in self.backbone.parameters():
            param.requires_grad_(False)

    def _prefix_token_count(self):
        if hasattr(self.backbone, 'num_prefix_tokens'):
            return int(self.backbone.num_prefix_tokens)
        storage_tokens = getattr(self.backbone, 'n_storage_tokens', None)
        if storage_tokens is None:
            storage_tokens = getattr(self.backbone, 'num_register_tokens', 0)
        return 1 + int(storage_tokens or 0)

    def _patch_grid(self, images):
        patch_size = getattr(self.backbone, 'patch_size', 16)
        if isinstance(patch_size, (tuple, list)):
            patch_h, patch_w = int(patch_size[0]), int(patch_size[1])
        else:
            patch_h = patch_w = int(patch_size)
        return images.shape[-2] // patch_h, images.shape[-1] // patch_w

    def _unwrap_tokens(self, prepared):
        if torch.is_tensor(prepared):
            return prepared
        if isinstance(prepared, (tuple, list)):
            for item in prepared:
                if torch.is_tensor(item):
                    return item
        if isinstance(prepared, dict):
            for key in ('x', 'tokens', 'last_hidden_state'):
                if key in prepared and torch.is_tensor(prepared[key]):
                    return prepared[key]
            for item in prepared.values():
                if torch.is_tensor(item):
                    return item
        raise TypeError(f'Cannot unwrap DINO token output of type {type(prepared)}')

    def _prepare_tokens_with_rope(self, images):
        try:
            prepared = self.backbone.prepare_tokens_with_masks(images, None)
        except TypeError:
            prepared = self.backbone.prepare_tokens_with_masks(images)
        tokens = self._unwrap_tokens(prepared)
        rope = None
        if hasattr(self.backbone, 'rope_embed'):
            grid_h, grid_w = self._patch_grid(images)
            try:
                rope = self.backbone.rope_embed(H=grid_h, W=grid_w)
            except TypeError:
                rope = self.backbone.rope_embed(grid_h, grid_w)
        return tokens, rope

    def _repeat_rope_for_query_context(self, rope):
        if rope is None or not torch.is_tensor(rope):
            return rope
        try:
            return rope.repeat_interleave(2, dim=-2)
        except RuntimeError:
            return rope

    def _run_blocks(self, tokens, rope):
        for block in self.backbone.blocks:
            if rope is None:
                tokens = block(tokens)
                continue
            try:
                tokens = block(tokens, rope=rope)
            except TypeError:
                try:
                    tokens = block(tokens, rope)
                except TypeError:
                    tokens = block(tokens)
        if hasattr(self.backbone, 'norm'):
            tokens = self.backbone.norm(tokens)
        return tokens

    def _forward_token_concat_features(self, query_images, context_images):
        query_tokens, query_rope = self._prepare_tokens_with_rope(query_images)
        context_tokens, _ = self._prepare_tokens_with_rope(context_images)
        prefix_tokens = self._prefix_token_count()
        pair_tokens = torch.cat([query_tokens[:, :prefix_tokens], query_tokens[:, prefix_tokens:], context_tokens[:, prefix_tokens:]], dim=1)
        pair_rope = self._repeat_rope_for_query_context(query_rope)
        pair_tokens = self._run_blocks(pair_tokens, pair_rope)
        return pair_tokens[:, 0]

    def _pool_backbone_features(self, images):
        features = self.backbone.forward_features(images)
        if isinstance(features, dict):
            if 'x_norm_clstoken' in features:
                return features['x_norm_clstoken']
            if 'x_norm_patchtokens' in features:
                return features['x_norm_patchtokens'].mean(dim=1)
            for item in features.values():
                if torch.is_tensor(item):
                    return item[:, 0] if item.ndim == 3 else item
        if torch.is_tensor(features):
            return features[:, 0] if features.ndim == 3 else features
        raise TypeError(f'Cannot pool DINO features of type {type(features)}')

    def _forward_feature_fusion(self, query_images, context_images):
        if context_images.ndim == 5:
            context_images = context_images[:, 0]
        query_features = self._pool_backbone_features(query_images)
        context_features = self._pool_backbone_features(context_images)
        self.active_fusion_mode = 'feature_fusion'
        return self.feature_fusion(torch.cat([query_features, context_features], dim=1))

    def _forward_topk_token_attention(self, query_images, context_images, context_scores=None):
        if context_images.ndim == 4:
            context_images = context_images.unsqueeze(1)
        batch_size, num_contexts, channels, height, width = context_images.shape
        query_repeated = query_images.unsqueeze(1).expand(-1, num_contexts, -1, -1, -1).reshape(batch_size * num_contexts, channels, height, width)
        context_flat = context_images.reshape(batch_size * num_contexts, channels, height, width)
        pair_features = self._forward_token_concat_features(query_repeated, context_flat).reshape(batch_size, num_contexts, self.feature_dim)
        query_summary = pair_features.mean(dim=1)
        attention_hidden = torch.tanh(self.context_proj(pair_features) + self.query_proj(query_summary).unsqueeze(1))
        attention_logits = self.context_attention(attention_hidden).squeeze(-1)
        if context_scores is not None:
            if context_scores.ndim == 1:
                context_scores = context_scores.unsqueeze(1)
            attention_logits = attention_logits + RETRIEVAL_SCORE_WEIGHT * context_scores.to(attention_logits.device, dtype=attention_logits.dtype)
        attention_weights = torch.softmax(attention_logits, dim=1)
        fused_features = torch.sum(pair_features * attention_weights.unsqueeze(-1), dim=1)
        fused_features = self.fusion_norm(fused_features)
        self.active_fusion_mode = 'topk_token_attention'
        return fused_features, attention_weights

    def forward(self, query_images, context_images, context_scores=None, return_features=False):
        try:
            fused_features, attention_weights = self._forward_topk_token_attention(query_images, context_images, context_scores=context_scores)
            logits = self.token_head(fused_features)
            if return_features:
                return logits, F.normalize(fused_features.float(), dim=1), attention_weights
            return logits
        except Exception as exc:
            if self.allow_feature_fallback:
                print(f'Top-k token attention failed, fallback to feature fusion: {type(exc).__name__}: {exc}')
                logits = self._forward_feature_fusion(query_images, context_images)
                if return_features:
                    return logits, F.normalize(logits.float(), dim=1), None
                return logits
            raise RuntimeError('Top-k token attention failed. Keep allow_feature_fallback=False for debugging.') from exc

ensure_dinov3_assets()

class DuplicateQueryContextDataset(Dataset):
    def __init__(self, annotations, query_transform, context_transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.query_transform = query_transform
        self.context_transform = context_transform
        self.crop_bbox = crop_bbox

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        image = self._open_image(row)
        query_image = self.query_transform(image.copy())
        context_image = self.context_transform(image.copy())
        return {
            'query_image': query_image,
            'context_image': context_image,
            'label': torch.tensor(int(row['class_id']), dtype=torch.long),
            'fname': row['fname'],
            'row_id': torch.tensor(int(row['row_id']), dtype=torch.long),
        }


train_dataset = DuplicateQueryContextDataset(train_df, train_transform, train_transform, crop_bbox=USE_BBOX_CROP)
val_dataset = DuplicateQueryContextDataset(val_df, eval_transform, eval_transform, crop_bbox=USE_BBOX_CROP)
test_dataset = DuplicateQueryContextDataset(test_df, eval_transform, eval_transform, crop_bbox=USE_BBOX_CROP)

loader_kwargs = {'num_workers': NUM_WORKERS, 'pin_memory': torch.cuda.is_available()}
if NUM_WORKERS > 0:
    loader_kwargs['persistent_workers'] = True
    loader_kwargs['prefetch_factor'] = 2

train_generator = torch.Generator()
train_generator.manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=train_generator, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

model = DinoV3FrozenSimilarityContextClassifier(
    DINOV3_REPO_DIR,
    DINOV3_MODEL_NAME,
    resolve_dinov3_weights(),
    NUM_CLASSES,
    FEATURE_DIM,
    allow_feature_fallback=False,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
backbone_params = [p for name, p in model.named_parameters() if name.startswith('backbone.')]
head_params = [p for name, p in model.named_parameters() if not name.startswith('backbone.')]
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR, 'weight_decay': BACKBONE_WEIGHT_DECAY, 'name': 'backbone'},
    {'params': head_params, 'lr': HEAD_LR, 'weight_decay': HEAD_WEIGHT_DECAY, 'name': 'head'},
])

warmup_epochs = max(1, min(WARMUP_EPOCHS, EPOCHS))
main_epochs = max(1, EPOCHS - warmup_epochs)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.2, end_factor=1.0, total_iters=warmup_epochs)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=main_epochs, eta_min=1e-7)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

checkpoint_path = OUT_PUT_DIR / 'final_duplicate_query_pairconcat_dinov3_vits16_sun397.pt'
last_checkpoint_path = OUT_PUT_DIR / 'checkpoint_last.pt'
best_checkpoint_path = OUT_PUT_DIR / 'best_val_checkpoint.pt'

torch.save({
    'method': 'Duplicate Query',
    'context_ablation_method': CONTEXT_ABLATION_METHOD,
    'selector_mode': 'none',
    'fusion_mode': 'strict_token_pair_concat',
    'seed': SEED,
    'use_bbox_crop': USE_BBOX_CROP,
    'candidate_pool_fraction_for_reference': CANDIDATE_POOL_FRACTION,
    'candidate_pool_positions_for_reference': torch.as_tensor(candidate_pool_positions, dtype=torch.long),
}, OUT_PUT_DIR / 'duplicate_query_context_config.pt')

sample = train_dataset[0]
print('Saved Duplicate Query config:', (OUT_PUT_DIR / 'duplicate_query_context_config.pt').resolve())
print('Method: Table 4 / Duplicate Query / strict token pair-concat')
print('Context tensor: same raw image as query, transformed independently')
print('Sample query tensor:', sample['query_image'].shape)
print('Initial fusion mode:', model.active_fusion_mode)
print(model)


## 5. Train Duplicate Query Baseline và lưu checkpoint

Mỗi batch dùng context được load từ cùng raw image với query, nhưng qua transform riêng. Loss gồm classification CE và SupCon regularization; không có selector, không có policy reward.


In [ ]:
from sklearn.metrics import f1_score


def macro_f1_from_predictions(labels, predictions):
    return float(f1_score(labels, predictions, average='macro', zero_division=0))


def supervised_contrastive_loss(features, labels, temperature=0.10):
    features = F.normalize(features.float(), dim=1)
    labels = labels.view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(features.device)
    logits = torch.matmul(features, features.T) / temperature
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    logits_mask = torch.ones_like(mask) - torch.eye(mask.size(0), device=features.device)
    positives_mask = mask * logits_mask
    positive_counts = positives_mask.sum(dim=1)
    valid = positive_counts > 0
    if not torch.any(valid):
        return features.sum() * 0.0
    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp_min(1e-12))
    mean_log_prob_pos = (positives_mask * log_prob).sum(dim=1) / positive_counts.clamp_min(1.0)
    return -mean_log_prob_pos[valid].mean()


def set_backbone_trainable(network, trainable):
    for name, param in network.named_parameters():
        if name.startswith('backbone.'):
            param.requires_grad_(trainable)
    if hasattr(network, 'backbone'):
        network.backbone.train(mode=trainable)


def current_lrs(opt):
    return {group.get('name', f'group_{index}'): group['lr'] for index, group in enumerate(opt.param_groups)}


def move_query_batch_to_device(batch):
    return {
        'query_images': batch['query_image'].to(DEVICE, non_blocking=True),
        'labels': batch['label'].to(DEVICE, non_blocking=True),
        'fname': batch['fname'],
        'context_images': batch['context_image'].to(DEVICE, non_blocking=True),
    }


def forward_duplicate_context(network, query_images, context_images, return_features=False):
    return network(
        query_images,
        context_images.unsqueeze(1),
        context_scores=None,
        return_features=return_features,
    )


def train_one_epoch(network, loader, epoch, backbone_trainable=True):
    network.train()
    set_backbone_trainable(network, backbone_trainable)
    running_loss = running_ce_loss = running_supcon_loss = 0.0
    running_correct = running_total = 0
    all_predictions, all_labels = [], []
    progress = tqdm(loader, desc=f'Epoch {epoch:03d} [train duplicate]', leave=False)

    for batch in progress:
        moved = move_query_batch_to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            logits, features, _attention = forward_duplicate_context(
                network,
                moved['query_images'],
                moved['context_images'],
                return_features=True,
            )
            ce_loss = criterion(logits, moved['labels'])
            supcon_loss = supervised_contrastive_loss(features, moved['labels'], temperature=SUPCON_TEMPERATURE)
            loss = ce_loss + SUPCON_WEIGHT * supcon_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(network.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        predictions = logits.argmax(dim=1)
        batch_size = moved['labels'].size(0)
        running_correct += (predictions == moved['labels']).sum().item()
        running_total += batch_size
        running_loss += loss.item() * batch_size
        running_ce_loss += ce_loss.item() * batch_size
        running_supcon_loss += supcon_loss.item() * batch_size
        all_predictions.extend(predictions.detach().cpu().tolist())
        all_labels.extend(moved['labels'].detach().cpu().tolist())
        progress.set_postfix(loss=running_loss / max(running_total, 1), acc=running_correct / max(running_total, 1))

    return {
        'loss': running_loss / max(running_total, 1),
        'ce_loss': running_ce_loss / max(running_total, 1),
        'supcon_loss': running_supcon_loss / max(running_total, 1),
        'accuracy': running_correct / max(running_total, 1),
        'macro_f1': macro_f1_from_predictions(all_labels, all_predictions),
        'context_mode': CONTEXT_ABLATION_METHOD,
    }


@torch.no_grad()
def evaluate_one_epoch(network, loader, desc='val'):
    network.eval()
    running_loss = running_correct = running_total = 0
    all_predictions, all_labels = [], []
    for batch in tqdm(loader, desc=desc, leave=False):
        moved = move_query_batch_to_device(batch)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            logits = forward_duplicate_context(
                network,
                moved['query_images'],
                moved['context_images'],
                return_features=False,
            )
            loss = criterion(logits, moved['labels'])
        predictions = logits.argmax(dim=1)
        batch_size = moved['labels'].size(0)
        running_loss += loss.item() * batch_size
        running_correct += (predictions == moved['labels']).sum().item()
        running_total += batch_size
        all_predictions.extend(predictions.detach().cpu().tolist())
        all_labels.extend(moved['labels'].detach().cpu().tolist())
    return {
        'loss': running_loss / max(running_total, 1),
        'accuracy': running_correct / max(running_total, 1),
        'macro_f1': macro_f1_from_predictions(all_labels, all_predictions),
        'context_mode': CONTEXT_ABLATION_METHOD,
    }


def build_checkpoint_payload(epoch, train_metrics, val_metrics, best_val_loss, best_epoch, history):
    return {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'class_names': class_names,
        'metrics': {
            'train_loss': train_metrics['loss'],
            'train_ce_loss': train_metrics.get('ce_loss'),
            'train_supcon_loss': train_metrics.get('supcon_loss'),
            'train_accuracy': train_metrics['accuracy'],
            'train_macro_f1': train_metrics['macro_f1'],
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
            'best_val_loss': best_val_loss,
            'best_epoch': best_epoch,
        },
        'history': history,
        'config': {
            'dataset_root': str(DATASET_ROOT),
            'method': 'Duplicate Query',
            'context_ablation_method': CONTEXT_ABLATION_METHOD,
            'selector_mode': 'none',
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'backbone_lr': BACKBONE_LR,
            'head_lr': HEAD_LR,
            'supcon_weight': SUPCON_WEIGHT,
            'use_bbox_crop': USE_BBOX_CROP,
            'image_size': IMAGE_SIZE,
            'dino_model': DINOV3_MODEL_NAME,
            'fusion_mode': 'strict_token_pair_concat_independent_duplicate_transform',
        },
    }


RESUME_FROM_LAST_CHECKPOINT = False
history = []
start_epoch = 1
best_val_loss = float('inf')
best_epoch = 0
bad_epochs = 0

if RESUME_FROM_LAST_CHECKPOINT and last_checkpoint_path.is_file():
    checkpoint = torch.load(last_checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint.get('history', [])
    metrics = checkpoint.get('metrics', {})
    best_val_loss = metrics.get('best_val_loss', min((row.get('val_loss', float('inf')) for row in history), default=float('inf')))
    best_epoch = metrics.get('best_epoch', 0)
    print(f'Resuming from epoch {start_epoch}')

for epoch in range(start_epoch, EPOCHS + 1):
    backbone_trainable = epoch > FREEZE_BACKBONE_EPOCHS
    train_metrics = train_one_epoch(model, train_loader, epoch, backbone_trainable=backbone_trainable)
    val_metrics = evaluate_one_epoch(model, val_loader, desc=f'Epoch {epoch:03d} [val duplicate]')
    scheduler.step()
    is_best = val_metrics['loss'] < best_val_loss - 1e-6
    if is_best:
        best_val_loss, best_epoch, bad_epochs = val_metrics['loss'], epoch, 0
    else:
        bad_epochs += 1
    lr_values = current_lrs(optimizer)
    record = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_ce_loss': train_metrics.get('ce_loss'),
        'train_supcon_loss': train_metrics.get('supcon_loss'),
        'train_accuracy': train_metrics['accuracy'],
        'train_macro_f1': train_metrics['macro_f1'],
        'val_loss': val_metrics['loss'],
        'val_accuracy': val_metrics['accuracy'],
        'val_macro_f1': val_metrics['macro_f1'],
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'bad_epochs': bad_epochs,
        'backbone_trainable': backbone_trainable,
        'backbone_lr': lr_values.get('backbone'),
        'head_lr': lr_values.get('head'),
        'fusion_mode': 'strict_token_pair_concat_independent_duplicate_transform',
        'context_mode': CONTEXT_ABLATION_METHOD,
    }
    history.append(record)
    payload = build_checkpoint_payload(epoch, train_metrics, val_metrics, best_val_loss, best_epoch, history)
    torch.save(payload, last_checkpoint_path)
    if is_best:
        torch.save(payload, best_checkpoint_path)
    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train_loss={train_metrics['loss']:.4f} ce={train_metrics.get('ce_loss', 0):.4f} "
        f"supcon={train_metrics.get('supcon_loss', 0):.4f} "
        f"train_acc={train_metrics['accuracy']:.4f} train_f1={train_metrics['macro_f1']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
        f"val_f1={val_metrics['macro_f1']:.4f} | "
        f"best_val={best_val_loss:.4f}@{best_epoch:03d} bad={bad_epochs}/{PATIENCE} | "
        f"bb_train={backbone_trainable} bb_lr={lr_values.get('backbone'):.2e} "
        f"head_lr={lr_values.get('head'):.2e}"
    )
    if bad_epochs >= PATIENCE:
        print(f'Early stopping at epoch {epoch}; best validation loss was {best_val_loss:.4f} at epoch {best_epoch}.')
        break

final_train_metrics = train_metrics
final_val_metrics = val_metrics
final_payload = build_checkpoint_payload(epoch, final_train_metrics, final_val_metrics, best_val_loss, best_epoch, history)
torch.save(final_payload, checkpoint_path)
print('Saved final checkpoint to:', checkpoint_path.resolve())
if best_checkpoint_path.is_file():
    print('Saved best validation checkpoint to:', best_checkpoint_path.resolve())
history_df = pd.DataFrame(history)
history_df.to_csv(OUT_PUT_DIR / 'training_history.csv', index=False)
display(history_df.tail())


## 6. Vẽ training history

Dùng để kiểm tra loss/accuracy trong quá trình train. Benchmark chính nằm ở cell `Official test accuracy`.


In [ ]:
import pandas as pd
if 'history_df' not in globals():
    history_df = pd.read_csv(OUT_PUT_DIR / 'training_history.csv')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_df.plot(x='epoch', y='train_loss', ax=axes[0], marker='o', markevery=max(1, len(history_df) // 10), title='Training loss')
history_df.plot(x='epoch', y='train_accuracy', ax=axes[1], marker='o', markevery=max(1, len(history_df) // 10), title='Training top-1 accuracy')
for axis in axes:
    axis.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
display(history_df.tail())


## 7. Official Test Accuracy - Duplicate Query Context

Cell này load checkpoint tốt nhất theo validation loss và đánh giá trên test split. File `test_predictions.csv` vẫn có các cột context để dễ so sánh với các notebook khác, nhưng context ở đây luôn là `<duplicate_query>`.


## 8. Inference và load checkpoint

Dùng hàm này để test nhanh một ảnh. Nếu không truyền context thủ công, selector TACS sẽ chọn context có utility score cao nhất từ candidate pool.


In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm


def macro_f1_score_np(y_true, y_pred, num_classes):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    f1_values = []
    for class_id in range(num_classes):
        tp = np.sum((y_true == class_id) & (y_pred == class_id))
        fp = np.sum((y_true != class_id) & (y_pred == class_id))
        fn = np.sum((y_true == class_id) & (y_pred != class_id))
        denom = 2 * tp + fp + fn
        f1_values.append(0.0 if denom == 0 else float((2 * tp) / denom))
    return float(np.mean(f1_values))


@torch.no_grad()
def evaluate_duplicate_context_model(network, loader, top_k=TOP_K):
    network.eval()
    total_loss = 0.0
    all_targets, all_predictions, rows = [], [], []
    top_k = min(top_k, NUM_CLASSES)

    def get_class_name(cid):
        return class_names[cid] if 0 <= cid < len(class_names) else f"class_{cid}"

    with torch.inference_mode():
        for batch in tqdm(loader, desc='test duplicate-query', leave=False):
            moved = move_query_batch_to_device(batch)
            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                logits = forward_duplicate_context(
                    network,
                    moved['query_images'],
                    moved['context_images'],
                    return_features=False,
                )
                loss = criterion(logits, moved['labels'])

            probs = torch.softmax(logits, dim=1).cpu()
            values, indices = torch.topk(probs, k=top_k, dim=1)
            predictions = logits.argmax(dim=1).detach().cpu()
            labels_cpu = moved['labels'].detach().cpu()

            total_loss += loss.item() * moved['query_images'].size(0)
            all_targets.extend(labels_cpu.tolist())
            all_predictions.extend(predictions.tolist())

            for fname, label, pred, value_row, index_row in zip(batch['fname'], labels_cpu, predictions, values, indices):
                label_id = int(label)
                pred_id = int(pred)
                top_class_ids = [int(index) for index in index_row]
                rows.append({
                    'fname': fname,
                    'true_class_id': label_id,
                    'true_car_name': get_class_name(label_id),
                    'context_fnames': '<duplicate_query>',
                    'context_class_ids': '',
                    'context_car_names': '<duplicate_query>',
                    'top1_context_fname': '<duplicate_query>',
                    'top1_context_class_id': -1,
                    'top1_context_car_name': '<duplicate_query>',
                    'top1_context_pool_position': -1,
                    'top1_tacs_score': 0.0,
                    'same_class_context_top1': False,
                    'same_class_context_any_topk': False,
                    'same_class_context': False,
                    'predicted_class_id': pred_id,
                    'predicted_car_name': get_class_name(pred_id),
                    'correct': bool(pred_id == label_id),
                    'confidence': float(value_row[0]),
                    'top_k_class_ids': '|'.join(str(index) for index in top_class_ids),
                })

    avg_loss = total_loss / len(loader.dataset)
    acc = float(np.mean([r['correct'] for r in rows])) if rows else 0.0
    macro_f1 = macro_f1_score_np(all_targets, all_predictions, NUM_CLASSES)
    return {
        'loss': avg_loss,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'context_mode': CONTEXT_ABLATION_METHOD,
        'predictions': pd.DataFrame(rows),
    }


eval_checkpoint_path = best_checkpoint_path if best_checkpoint_path.is_file() else checkpoint_path
print('Loaded checkpoint for evaluation:', eval_checkpoint_path.resolve())
checkpoint = torch.load(eval_checkpoint_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

if 'class_names' not in globals() or len(class_names) != NUM_CLASSES:
    class_names = load_class_names(DATASET_ROOT)

results = evaluate_duplicate_context_model(model, test_loader, top_k=TOP_K)
test_prediction_df = results.pop('predictions')
test_prediction_df.to_csv(OUT_PUT_DIR / 'test_predictions.csv', index=False)

metrics = results
print(f"context_mode: {metrics['context_mode']}")
print(f"test_loss: {metrics['loss']:.4f}")
print(f"test_accuracy: {metrics['accuracy']:.4f}")
print(f"test_macro_f1: {metrics['macro_f1']:.4f}")

display(test_prediction_df.head())


In [ ]:
def load_duplicate_context_model_for_inference(checkpoint_file, device=DEVICE):
    saved = torch.load(checkpoint_file, map_location=device)
    loaded_model = DinoV3FrozenSimilarityContextClassifier(
        repo_dir=DINOV3_REPO_DIR,
        model_name=DINOV3_MODEL_NAME,
        weights_path=resolve_dinov3_weights(),
        num_classes=len(saved['class_names']),
        feature_dim=FEATURE_DIM,
        allow_feature_fallback=False,
    )
    loaded_model.load_state_dict(saved['model_state_dict'])
    return loaded_model.to(device).eval(), saved['class_names']


model, class_names = load_duplicate_context_model_for_inference(best_checkpoint_path)


@torch.no_grad()
def predict_one_image_with_duplicate_context(image_path, network=model, top_k=TOP_K):
    if network is None:
        network = model
    image_path = Path(image_path)
    pil_image = Image.open(image_path).convert('RGB')
    query_image = eval_transform(pil_image.copy()).unsqueeze(0).to(DEVICE)
    context_image = eval_transform(pil_image.copy()).unsqueeze(0).to(DEVICE)

    network.eval()
    with torch.inference_mode():
        logits = forward_duplicate_context(network, query_image, context_image, return_features=False)
        probs = torch.softmax(logits, dim=1)[0]
        values, indices = torch.topk(probs, k=min(top_k, NUM_CLASSES))

    print('Query:', image_path)
    print('Context: <duplicate_query>')
    print('\nTop predictions:')
    for rank, (class_id, probability) in enumerate(zip(indices.tolist(), values.tolist()), start=1):
        print(f'{rank}. {class_names[class_id]} | p={probability:.4f}')


print('Final checkpoint:', checkpoint_path.resolve())
print('Best-val checkpoint:', best_checkpoint_path.resolve())
print('Latest checkpoint:', last_checkpoint_path.resolve())
print('Method:', 'Table 4 - Duplicate Query context / strict token pair-concat / independent transforms')


In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

## 9. Ghi chú benchmark

- Method này là hàng **Duplicate Query** trong Table 4.
- Mỗi query được ghép với cùng raw image của chính nó, nhưng query/context đi qua transform riêng.
- Không dùng `context = query.clone()` sau augmentation, vì cách đó làm mô hình nhìn đúng cùng token hai lần và có thể tạo shortcut.
- Bản này dùng **strict token pair-concat**, không dùng gated query branch, để giữ ablation sạch.
- Notebook không dùng candidate retrieval, không train selector, không dùng Gumbel-Softmax và không dùng policy reward.
- Metric chính cho SUN là `test_accuracy`; `same_class_context_rate` không có ý nghĩa với duplicate-query context.


In [ ]:

# --- CELL CUỐI: TỰ ĐỘNG DỪNG TÍNH TOÁN & SHUTDOWN PHIÊN ---
import os
import time
import torch

print("=== QUÁ TRÌNH HUẤN LUYỆN VÀ ĐÁNH GIÁ ĐÃ HOÀN TẤT ===")

# 1. Giải phóng toàn bộ VRAM
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(2)

# 2. Tắt Kernel để dừng ngay lập tức mọi tác vụ đang chạy
try:
    from ipykernel.kernelapp import IPKernelApp
    IPKernelApp.instance().shutdown()
except Exception:
    os._exit(0)

In [ ]:
# import numpy as np
# import pandas as pd
# from PIL import Image
# from tqdm import tqdm

# def analyze_dataset_dimensions(df, check_bbox=True):
#     widths, heights = [], []
#     crop_w_list, crop_h_list = [], []

#     for _, row in tqdm(df.iterrows(), total=len(df), desc="Đang quét kích thước ảnh"):
#         # Image.open chỉ đọc metadata header, cực nhanh và không tốn RAM
#         with Image.open(row['image_path']) as img:
#             w, h = img.size
#             widths.append(w)
#             heights.append(h)

#         if check_bbox and 'bbox_x1' in row:
#             left = max(0, int(row['bbox_x1']) - 1)
#             top = max(0, int(row['bbox_y1']) - 1)
#             right = max(left + 1, int(row['bbox_x2']))
#             bottom = max(top + 1, int(row['bbox_y2']))
#             crop_w_list.append(right - left)
#             crop_h_list.append(bottom - top)

#     data = {
#         "Metric": ["Min", "Max", "Mean", "Median"],
#         "Orig Width (W)": [np.min(widths), np.max(widths), round(np.mean(widths), 1), np.median(widths)],
#         "Orig Height (H)": [np.min(heights), np.max(heights), round(np.mean(heights), 1), np.median(heights)],
#     }

#     if check_bbox and crop_w_list:
#         data["Crop Width (BBox)"] = [np.min(crop_w_list), np.max(crop_w_list), round(np.mean(crop_w_list), 1), np.median(crop_w_list)]
#         data["Crop Height (BBox)"] = [np.min(crop_h_list), np.max(crop_h_list), round(np.mean(crop_h_list), 1), np.median(crop_h_list)]

#     stats_df = pd.DataFrame(data)
    
#     # Tính tỉ lệ khung hình (Aspect Ratio W/H)
#     aspect_ratios = np.array(widths) / np.array(heights)
#     print(f"Aspect ratio gốc trung bình (W/H): {aspect_ratios.mean():.2f}")
#     print(f"Số ảnh có W > H (ảnh ngang): {np.sum(aspect_ratios > 1.0)} / {len(df)} ({np.mean(aspect_ratios > 1.0):.1%})")
    
#     return stats_df

# # Kiểm tra trên tập train
# train_size_stats = analyze_dataset_dimensions(full_train_df, check_bbox=USE_BBOX_CROP)
# display(train_size_stats)

In [ ]:
# --- CELL DỪNG NOTEBOOK & GIẢI PHÓNG TÀI NGUYÊN ---
import gc
import os
import time
import torch

print("=== HOÀN TẤT: ĐANG DỌN TÀI NGUYÊN VÀ DỪNG PHIÊN ===")

# 1. Thu hồi bộ nhớ RAM và VRAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(2)

# 2. Dừng và đóng Kernel ngay lập tức
try:
    from ipykernel.kernelapp import IPKernelApp
    IPKernelApp.instance().shutdown()
except Exception:
    os._exit(0)